In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import os, time, copy, numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import datasets, transforms, models
from PIL import Image

FOLDER_NAME = "tumor dataset 1316/data"

DATA_DIR = f"/content/drive/MyDrive/{FOLDER_NAME}"
OUTPUT_DIR = f"/content/drive/MyDrive/BrainTumorModel"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 8
LR = 1e-4
WEIGHT_DECAY = 1e-4
SEED = 42
NUM_WORKERS = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Using dataset path:", DATA_DIR)

torch.manual_seed(SEED)
np.random.seed(SEED)


train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tfms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.1)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# build dataset
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"Dataset folder not found: {DATA_DIR}")

full_dataset = datasets.ImageFolder(DATA_DIR)
class_names = full_dataset.classes
NUM_CLASSES = len(class_names)
print("Classes found:", class_names)
print("Total images:", len(full_dataset))

if NUM_CLASSES < 2:
    raise ValueError(
        f"Only {NUM_CLASSES} class found. For classification you need at least 2 classes.\n"
        "Check that you pointed DATA_DIR to the folder that directly contains class subfolders.\n"
        "Example correct path: '/content/drive/MyDrive/tumor dataset/data' (not '/content/drive/MyDrive/tumor dataset')."
    )


targets = [s[1] for s in full_dataset.samples]
train_idx, val_idx = train_test_split(
    np.arange(len(full_dataset)),
    test_size=0.2,
    stratify=targets,
    random_state=SEED
)


class CustomImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


train_samples = [full_dataset.samples[i] for i in train_idx]
val_samples = [full_dataset.samples[i] for i in val_idx]

train_dataset = CustomImageDataset(train_samples, transform=train_tfms)
val_dataset   = CustomImageDataset(val_samples, transform=val_tfms)


train_labels = [targets[i] for i in train_idx]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
print("Train class counts:", dict(zip(class_names, class_counts)))
class_weights = 1.0 / (class_counts + 1e-6)
sample_weights = torch.DoubleTensor([class_weights[t] for t in train_labels])
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


if device == torch.device("cpu"):
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2)


def train_one_epoch():
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_probs = []
    all_targets = []
    for imgs, labels in tqdm(train_loader, desc="Train", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        probs = torch.softmax(outputs.detach().cpu(), dim=1).numpy()
        preds = np.argmax(probs, axis=1)
        all_probs.append(probs)
        all_targets.append(labels.detach().cpu().numpy())
        correct += (preds == labels.detach().cpu().numpy()).sum()
        total += imgs.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    train_acc = correct / total if total > 0 else 0.0
    all_probs = np.vstack(all_probs)
    all_targets = np.concatenate(all_targets)

    try:
        train_auc = roc_auc_score(all_targets, all_probs, multi_class='ovr')
    except Exception:
        train_auc = 0.0
    return train_loss, train_acc, train_auc

def validate():
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="Val", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * imgs.size(0)
            probs = torch.softmax(outputs.cpu(), dim=1).numpy()
            preds = np.argmax(probs, axis=1)
            all_probs.append(probs)
            all_targets.append(labels.cpu().numpy())
            correct += (preds == labels.cpu().numpy()).sum()
            total += imgs.size(0)

    val_loss = running_loss / len(val_loader.dataset)
    val_acc = correct / total if total > 0 else 0.0
    all_probs = np.vstack(all_probs)
    all_targets = np.concatenate(all_targets)
    try:
        val_auc = roc_auc_score(all_targets, all_probs, multi_class='ovr')
    except Exception:
        val_auc = 0.0
    return val_loss, val_acc, val_auc, all_probs, all_targets


best_auc = 0.0
best_state = None

for epoch in range(1, NUM_EPOCHS+1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    t0 = time.time()
    train_loss, train_acc, train_auc = train_one_epoch()
    val_loss, val_acc, val_auc, val_probs, val_targets = validate()
    scheduler.step(val_auc)


    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Train AUC: {train_auc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc*100:.2f}% | Val   AUC: {val_auc:.4f}")
    print(f"Epoch time: {time.time()-t0:.0f}s")


    if val_auc > best_auc:
        best_auc = val_auc
        best_state = copy.deepcopy(model.state_dict())
        ckpt = os.path.join(OUTPUT_DIR, f"best_resnet50_epoch{epoch}_auc{val_auc:.4f}.pth")
        torch.save(best_state, ckpt)
        print("Saved best checkpoint to:\n", ckpt)


if best_state is not None:
    model.load_state_dict(best_state)


from sklearn.metrics import confusion_matrix, classification_report
model.eval()
all_probs = []
all_targets = []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs.cpu(), dim=1).numpy()
        all_probs.append(probs)
        all_targets.append(labels.numpy())

all_probs = np.vstack(all_probs)
all_targets = np.concatenate(all_targets)
preds = np.argmax(all_probs, axis=1)
final_acc = (preds == all_targets).mean()
print("\n=== FINAL VALIDATION ===")
print(f"Final Accuracy: {final_acc*100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(all_targets, preds))
print("Classification Report:")
print(classification_report(all_targets, preds, target_names=class_names))
print("Best validation AUC achieved:", best_auc)


Using device: cpu
Using dataset path: /content/drive/MyDrive/tumor dataset 1316/data
Classes found: ['glioma', 'meningioma', 'notumor', 'pituitary']
Total images: 1311
Train class counts: {'glioma': np.int64(240), 'meningioma': np.int64(244), 'notumor': np.int64(324), 'pituitary': np.int64(240)}


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 1/8


Train Loss: 0.5327 | Train Acc: 81.11% | Train AUC: 0.9540
Val   Loss: 0.2832 | Val   Acc: 88.97% | Val   AUC: 0.9901
Epoch time: 871s
Saved best checkpoint to:
 /content/drive/MyDrive/BrainTumorModel/best_resnet50_epoch1_auc0.9901.pth

Epoch 2/8


Train Loss: 0.1882 | Train Acc: 94.37% | Train AUC: 0.9912
Val   Loss: 0.1851 | Val   Acc: 93.92% | Val   AUC: 0.9967
Epoch time: 858s
Saved best checkpoint to:
 /content/drive/MyDrive/BrainTumorModel/best_resnet50_epoch2_auc0.9967.pth

Epoch 3/8


...

Train Loss: 0.0802 | Train Acc: 97.52% | Train AUC: 0.9985
Val   Loss: 0.0952 | Val   Acc: 95.82% | Val   AUC: 0.9983
Epoch time: 839s
Saved best checkpoint to:
 /content/drive/MyDrive/BrainTumorModel/best_resnet50_epoch4_auc0.9983.pth

Epoch 5/8


Train Loss: 0.0868 | Train Acc: 97.61% | Train AUC: 0.9977
Val   Loss: 0.1373 | Val   Acc: 95.44% | Val   AUC: 0.9957
Epoch time: 847s

Epoch 6/8


Train Loss: 0.0759 | Train Acc: 97.61% | Train AUC: 0.9984
Val   Loss: 0.1108 | Val   Acc: 96.96% | Val   AUC: 0.9984
Epoch time: 860s
Saved best checkpoint to:
 /content/drive/MyDrive/BrainTumorModel/best_resnet50_epoch6_auc0.9984.pth

Epoch 7/8


Train Loss: 0.0696 | Train Acc: 97.52% | Train AUC: 0.9985
Val   Loss: 0.0871 | Val   Acc: 96.20% | Val   AUC: 0.9984
Epoch time: 853s
Saved best checkpoint to:
 /content/drive/MyDrive/BrainTumorModel/best_resnet50_epoch7_auc0.9984.pth

Epoch 8/8


Train Loss: 0.0462 | Train Acc: 99.14% | Train AUC: 0.9987
Val   Loss: 0.0990 | Val   Acc: 95.06% | Val   AUC: 0.9979
Epoch time: 856s

=== FINAL VALIDATION ===
Final Accuracy: 96.20%
Confusion Matrix:
[[58  2  0  0]
 [ 1 57  0  4]
 [ 0  2 79  0]
 [ 1  0  0 59]]
Classification Report:
              precision    recall  f1-score   support

      glioma       0.97      0.97      0.97        60
  meningioma       0.93      0.92      0.93        62
     notumor       1.00      0.98      0.99        81
   pituitary       0.94      0.98      0.96        60

    accuracy                           0.96       263
   macro avg       0.96      0.96      0.96       263
weighted avg       0.96      0.96      0.96       263

Best validation AUC achieved: 0.9984155165738904
